# Financial Engineering Deep Research Agent

A self-contained Gradio application that uses OpenAI Agents to plan, execute, and synthesize deep research on complex Financial Engineering topics.

In [ ]:
import asyncio
import gradio as gr
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from agents import Agent, WebSearchTool, ModelSettings, Runner

load_dotenv(override=True)

In [2]:

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the financial query.")
    query: str = Field(description="The search term to use for the web search, focusing on mathematical and financial accuracy.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence executive summary of the financial findings.")
    markdown_report: str = Field(description="The final detailed report including solutions, formulas, and references.")
    follow_up_questions: list[str] = Field(description="Suggested financial engineering topics to research further.")

In [3]:
HOW_MANY_SEARCHES = 3

planner_agent = Agent(
    name="FinancialPlannerAgent",
    instructions=(
        "You are a Senior Quantitative Analyst and Financial Engineer. "
        f"Given a complex financial engineering query, devise a plan of {HOW_MANY_SEARCHES} targeted web searches "
        "to gather mathematical models, stochastic calculus principles, market data, and pricing formulas required to solve the problem."
    ),
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

search_agent = Agent(
    name="FinancialSearchAgent",
    instructions=(
        "You are a financial researcher. Given a search term related to financial engineering, quantitative finance, or mathematics, "
        "search the web and produce a highly detailed, accurate summary of the results. Capture the mathematical formulas, "
        "models, data, and references used. Ensure technical accuracy. The summary must be under 300 words, strictly capturing the core essence without fluff."
    ),
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

writer_agent = Agent(
    name="FinancialWriterAgent",
    instructions=(
        "You are an expert Financial Engineer and Quantitative Analyst writing a comprehensive report for another quant. "
        "You are provided with an original financial engineering query and search results compiled by your assistants.\n"
        "Write a lengthy and detailed solution in markdown format. Be sure to include:\n"
        "1. An outline of the approach.\n"
        "2. Rigorous mathematical and financial definitions.\n"
        "3. The proposed solution or theoretical framework.\n"
        "4. Proper references to the sources found in the search results."
    ),
    model="gpt-4o-mini",
    output_type=ReportData,
)

In [4]:

class FinancialResearchManager:
    async def run(self, query: str):
        yield "Planning financial searches..."
        search_plan = await self.plan_searches(query)
        
        yield f"Searches planned ({len(search_plan.searches)} queries), starting web searches..."
        search_results = await self.perform_searches(search_plan)
        
        yield "Searches completed, synthesizing mathematical and financial data into report..."
        report = await self.write_report(query, search_results)
        
        yield "Report generated successfully.\n\n---\n\n" + report.markdown_report

    async def plan_searches(self, query: str) -> WebSearchPlan:
        result = await Runner.run(
            planner_agent,
            f"Query: {query}",
        )
        return result.final_output_as(WebSearchPlan)

    async def perform_searches(self, search_plan: WebSearchPlan) -> list[str]:
        tasks = [asyncio.create_task(self.search(item)) for item in search_plan.searches]
        results = []
        for task in asyncio.as_completed(tasks):
            result = await task
            if result is not None:
                results.append(result)
        return results

    async def search(self, item: WebSearchItem) -> str | None:
        input_text = f"Search term: {item.query}\nReason for searching: {item.reason}"
        try:
            result = await Runner.run(search_agent, input_text)
            return str(result.final_output)
        except Exception as e:
            print(f"Search error for {item.query}: {e}")
            return None

    async def write_report(self, query: str, search_results: list[str]) -> ReportData:
        input_text = f"Original query: {query}\nSummarized search results: {search_results}"
        result = await Runner.run(writer_agent, input_text)
        return result.final_output_as(ReportData)

In [ ]:
async def research_in_ui(query: str):
    async for chunk in FinancialResearchManager().run(query):
        yield chunk

with gr.Blocks(theme=gr.themes.Monochrome()) as app:
    gr.Markdown("# 📈 QuantAgent: Financial Engineering Deep Research")
    gr.Markdown("Ask complex financial mathematical questions (e.g. 'Derive the Black-Scholes PDE and give examples of its numerical computation').")
    
    query_input = gr.Textbox(label="Research Query", placeholder="Enter your financial engineering question here...")
    run_button = gr.Button("Execute Deep Research", variant="primary")
    report_output = gr.Markdown(label="Research Progress & Final Report")

    run_button.click(fn=research_in_ui, inputs=query_input, outputs=report_output)
    query_input.submit(fn=research_in_ui, inputs=query_input, outputs=report_output)

# Launch the application
app.launch(inbrowser=True)